# Mini-Project 5 — Audit an MCP Server
### IT7075: Applied AI for Cybersecurity · Model Context Protocol

MCP is a contract. A server advertises tools; a client — or an LLM agent — discovers them and calls
them, trusting whatever comes back. So the interesting question is not *can you build a server*, it
is **what happens when the tool is quietly wrong**.

You have a working MCP server with a recon tool, a CVE-lookup tool, and a knowledge-base resource.
It has real defects. Your job is to find them by measurement, fix what code can fix, and say what
code cannot.

**MCP has two sides, so your five TODOs are split across two files — 12 lines in total:**

| | Where | What | Lines |
|---|---|---|---|
| **TODO 1** | `project_mcp_server.py` | a version-aware `lookup_cve_v2` tool | 2 |
| **TODO 2** | `project_mcp_server.py` | expose the KB as a **resource** | 1 |
| **TODO 3** | this notebook | connect over stdio and **discover** the tools | 3 |
| **TODO 4** | this notebook | **call** a tool | 3 |
| **TODO 5** | this notebook | **read** the resource | 3 |

Find them by searching for `###################` in **both** files.

> **Runs fully offline.** Simulated scans, a local knowledge base, no LLM — so no API key, no
> model, no GPU, and no packets on the network. The numbers you report are exact.

> **The server restarts every time.** The client launches `project_mcp_server.py` as a fresh
> subprocess on each connection, so edit the server file, re-run a client cell, and your change is
> live. Nothing to restart.

## Part 1 — Setup: running async MCP code inside Jupyter

MCP's client API is asynchronous, and Jupyter already owns an event loop. `run_async` — the helper
from the Lecture 3 notebook — runs a coroutine on a worker thread with its own loop. Given to you;
nothing to change.

In [26]:
#print("Google Colab is working!")
%pip install -q "mcp>=1,<2"

In [27]:
from mcp.server.fastmcp import FastMCP

print("MCP successfully imported!")

MCP successfully imported!


In [28]:
print("Testing Colab runtime...")
from mcp.server.fastmcp import FastMCP
print("MCP is ready!")

Testing Colab runtime...
MCP is ready!


In [29]:
import sys, os, json, re, asyncio, threading
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client


def run_async(coro):
    """GIVEN (Lecture 3): run a coroutine to completion on a worker thread,
    because Jupyter already owns the main event loop."""
    box = {}

    def worker():
        loop = asyncio.new_event_loop()
        asyncio.set_event_loop(loop)
        try:
            box["value"] = loop.run_until_complete(coro)
        except Exception as e:                       # noqa: BLE001
            box["error"] = e
        finally:
            loop.close()

    t = threading.Thread(target=worker)
    t.start()
    t.join()
    if "error" in box:
        raise box["error"]
    return box["value"]


# ---- where things live ---------------------------------------------------
def project_path(name):
    """The project files sit next to this notebook, or one level up from solutions/."""
    return name if os.path.exists(name) else os.path.join("..", name)


SERVER_FILE = project_path("project_mcp_server.py")
EXPECTED_FILE = project_path("expected_cves.json")

# How the client starts the server: as a subprocess, over stdio.
server = StdioServerParameters(command=sys.executable, args=[SERVER_FILE])
ERRLOG = open("mcp_server_stderr.log", "w")     # a real file: Jupyter dislikes pipes

print("server file :", SERVER_FILE)
print("python      :", sys.executable)

server file : project_mcp_server.py
python      : /usr/bin/python3


## Part 2 — Connect and discover  ·  **TODO 3**

This is the MCP handshake, and it is the whole point of the protocol: the client does not know what
the server can do until it asks. Read the descriptions it prints — that text is the entire contract
an LLM would use to decide which tool to call.

> **Expected output:** four tools — `port_scan`, `list_targets`, `lookup_cve`, `lookup_cve_v2`.

In [30]:
###################  TODO 3 — connect over stdio and list the tools  ###################
# WHAT: finish discover() so it returns [(tool name, first line of its docstring), ...]
#
# HOW (the Lecture 3 "Connect & Discover" cell) -- three lines inside the two
# `async with` blocks that are already open for you:
#
#   1. await session.initialize()                  the MCP handshake
#   2. tools = await session.list_tools()          ask what the server exposes
#   3. return [(t.name, t.description.splitlines()[0]) for t in tools.tools]
#
# Note `tools.tools`: list_tools() returns a result OBJECT with a .tools list on it.
################


async def discover():
    async with stdio_client(server, errlog=ERRLOG) as (read, write):
        async with ClientSession(read, write) as session:

            # Step 1: Initialize the MCP connection.
            await session.initialize()

            # Step 2: Ask the server which tools it offers.
            tools = await session.list_tools()

            # Step 3: Return tool names and descriptions.
            return [
                (t.name, t.description.splitlines()[0])
                for t in tools.tools
            ]


for name, description in run_async(discover()):
    print("- %-16s %s" % (name, description))


- port_scan        Run a (simulated) network port scan of a target host and return open ports
- list_targets     List the demo hosts this server can scan (simulated lab environment).
- lookup_cve       Look up known vulnerabilities for a service or product name (e.g.
- lookup_cve_v2    Look up known vulnerabilities for a service, matching on PRODUCT and


## Part 3 — Call a tool and read the resource  ·  **TODO 4 and TODO 5**

Two different MCP primitives. A **tool** is something the agent *does* (`call_tool`); a **resource**
is something it *reads* (`read_resource`). Same server, same session, different verbs.

> **Expected output:** a scan of `10.0.0.5` listing vsftpd, OpenSSH and Apache; then the first lines
> of the CVE knowledge base.

In [32]:
###################  TODO 4 — call a tool  ###################
# WHAT: run the `port_scan` tool on `target` and keep its text in `scan`.
#
# HOW (the Lecture 3 "Call the Tools" cell):
#       result = await session.call_tool("port_scan", {"target": target})
#       scan   = result.content[0].text
#
# The second argument is a DICT of the tool's parameters, keyed by name -- exactly
# the names in the server's function signature. Getting one wrong is the most common
# MCP client bug, and the server will tell you so.
################

###################  TODO 5 — read a resource  ###################
# WHAT: fetch the knowledge base the server exposes at the URI "kb://cve" into `kb`.
#
# HOW:  kb = await session.read_resource("kb://cve")
#       ...and the text is then kb.contents[0].text (see the return line below).
#
# Note the difference from a tool call: read_resource takes a URI and no arguments,
# and the payload is under .contents (plural) rather than .content.
################

async def scan_and_read(target):
    async with stdio_client(server, errlog=ERRLOG) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()

            # TODO 4: Call the port_scan tool
            result = await session.call_tool("port_scan", {"target": target})
            scan = result.content[0].text

            # TODO 5: Read the CVE knowledge base resource
            kb = await session.read_resource("kb://cve")

            return scan, kb.contents[0].text


scan_text, kb_text = run_async(scan_and_read("10.0.0.5"))

print(scan_text)
print("\nRESOURCE kb://cve — first 180 characters:")
print(kb_text[:180], "...")

Nmap scan report for 10.0.0.5
PORT     STATE SERVICE  VERSION
21/tcp   open  ftp      vsftpd 2.3.4
22/tcp   open  ssh      OpenSSH 7.2p2 Ubuntu
80/tcp   open  http     Apache httpd 2.4.49

RESOURCE kb://cve — first 180 characters:
# vsftpd 2.3.4
The vsftpd 2.3.4 FTP server (commonly seen on port 21) shipped with a malicious backdoor introduced into the source distribution. Connecting with a username ending i ...


## Part 4 — Sweep the lab  *(no TODO — run it and read it)*

The coding is done; everything from here is measurement. Scan all four hosts, then ask **both**
lookup tools about every service that turned up.

> **Expected output:** 11 services across the four hosts.

In [33]:

import re

# Match service information from simulated Nmap output.
SERVICE_LINE = re.compile(r"^\s*(\d+)/tcp\s+open\s+(\S+)\s+(.*)$")


def parse_services(scan_text):
    """Extract (port, service, version-banner) from Nmap-style output."""
    found = []

    for line in scan_text.splitlines():
        m = SERVICE_LINE.match(line)

        if m:
            found.append((
                m.group(1),
                m.group(2),
                m.group(3).strip()
            ))

    return found


async def sweep(hosts, tool_names):
    """Connect to the MCP server, scan hosts, and query each CVE lookup tool."""
    out = []

    async with stdio_client(server, errlog=ERRLOG) as (read, write):
        async with ClientSession(read, write) as session:

            await session.initialize()

            for host in hosts:

                # Step 1: Scan the simulated host.
                result = await session.call_tool(
                    "port_scan",
                    {"target": host}
                )

                scan = result.content[0].text

                # Step 2: Extract discovered services.
                for port, svc, banner in parse_services(scan):

                    row = {
                        "host": host,
                        "port": port,
                        "banner": banner
                    }

                    # Step 3: Query each vulnerability lookup tool.
                    for tool in tool_names:

                        res = await session.call_tool(
                            tool,
                            {"service": banner}
                        )

                        row[tool] = res.content[0].text

                    out.append(row)

    return out


# Include the four original hosts and our new simulated host.
HOSTS = [
    "10.0.0.5",
    "10.0.0.8",
    "10.0.0.12",
    "10.0.0.20",
    "10.0.0.30"
]

# Execute the vulnerability lookup sweep.
rows = run_async(
    sweep(HOSTS, ["lookup_cve", "lookup_cve_v2"])
)

print(
    "discovered %d services across %d hosts\n"
    % (len(rows), len(HOSTS))
)

for r in rows:
    print(
        "  %-10s %-5s %s"
        % (r["host"], r["port"], r["banner"])
    )


discovered 11 services across 5 hosts

  10.0.0.5   21    vsftpd 2.3.4
  10.0.0.5   22    OpenSSH 7.2p2 Ubuntu
  10.0.0.5   80    Apache httpd 2.4.49
  10.0.0.8   139   Samba smbd 3.5.0
  10.0.0.8   3306  MySQL 5.5.40
  10.0.0.12  23    Linux telnetd
  10.0.0.12  443   Apache httpd 2.2.8 (OpenSSL 1.0.1)
  10.0.0.12  6379  Redis key-value store 4.0.9
  10.0.0.20  21    ProFTPD 1.3.5
  10.0.0.20  80    nginx 1.18.0
  10.0.0.20  5432  PostgreSQL 9.6.24


In [35]:

import importlib
import project_mcp_server

importlib.reload(project_mcp_server)

# Check that our new simulated host exists.
host = "10.0.0.30"

assert host in project_mcp_server._SIMULATED_HOSTS

# Run a simulated scan directly.
scan_result = project_mcp_server.port_scan(host)

print(scan_result)


MCP successfully imported!
PASS: Server file has valid Python syntax
OpenSSH 7.2p2 Ubuntu: MATCH
Redis key-value store 4.0.9: MATCH
Linux telnetd: MATCH
Apache httpd 2.2.8: NO MATCH
Knowledge base loaded successfully!
Number of characters: 5435

First 400 characters:

# vsftpd 2.3.4
The vsftpd 2.3.4 FTP server (commonly seen on port 21) shipped with a malicious backdoor introduced into the source distribution. Connecting with a username ending in ":)" opens a command shell on TCP port 6200, granting remote command execution as root. Tracked as CVE-2011-2523. Severity: critical. Remediation: upgrade to a clean, vendor-verified vsftpd build and verify package che
PASS: CVE knowledge base contains expected information.
Nmap scan report for 10.0.0.30
PORT     STATE SERVICE  VERSION
21/tcp   open  ftp      vsftpd 2.3.4
22/tcp   open  ssh      OpenSSH 7.2p2 Ubuntu
8080/tcp open  http     Apache Tomcat 9.0.30


## Part 5 — Score both tools  *(no TODO)*

`expected_cves.json` records, for every service in the lab, which knowledge-base section *should*
come back — or `null` where the KB genuinely has no entry. Comparing each tool's answer with that
sorts every response into one of four buckets:

| | Meaning | Why it matters |
|---|---|---|
| **grounded** | the right advisory came back | what you want |
| **blind spot** | "no known CVEs" — but the KB *does* cover it | the agent is reassured, wrongly |
| **wrong version** | an advisory for a *different* version | worse than silence: it looks authoritative |
| **true gap** | "no known CVEs" and the KB really has nothing | honest — fix it with knowledge, not code |

> **Expected output:**
>
> | tool | grounded | blind spot | wrong version | true gap |
> |---|---:|---:|---:|---:|
> | `lookup_cve` (the lecture's) | 6/9 | **2** | **1** | 2 |
> | `lookup_cve_v2` (yours) | **9/9** | 0 | 0 | 2 |
>
> (11 services in all: 9 the knowledge base can cover, plus 2 it genuinely cannot.)
>
> Look at *which* ones the lecture version got wrong. `Linux telnetd` and
> `Redis key-value store 4.0.9` were **already in the knowledge base** — it simply failed to match
> them, and told the agent there was nothing to worry about. And `Apache httpd 2.2.8` came back with
> the advisory for **2.4.49**, a version that host is not running.
>
> Note *how* that last one fails. The naive tool returns the right OpenSSL advisory **and** the
> 2.4.49 one together — so it is not simply wrong, it is right with a fabricated extra finding
> attached. An analyst chasing a path-traversal bug on a server that cannot have it has lost just as
> much time as one who missed something.
>
> Your version fixes all three, and the two remaining gaps (nginx, PostgreSQL) are honest: the
> knowledge base has no entry. **No amount of matching logic fixes those** — that is Part 6.

In [37]:

import importlib
import project_mcp_server

importlib.reload(project_mcp_server)

result = project_mcp_server.summarize_host_security("10.0.0.30")

print(result)


MCP successfully imported!
PASS: Server file has valid Python syntax
OpenSSH 7.2p2 Ubuntu: MATCH
Redis key-value store 4.0.9: MATCH
Linux telnetd: MATCH
Apache httpd 2.2.8: NO MATCH
Knowledge base loaded successfully!
Number of characters: 5435

First 400 characters:

# vsftpd 2.3.4
The vsftpd 2.3.4 FTP server (commonly seen on port 21) shipped with a malicious backdoor introduced into the source distribution. Connecting with a username ending in ":)" opens a command shell on TCP port 6200, granting remote command execution as root. Tracked as CVE-2011-2523. Severity: critical. Remediation: upgrade to a clean, vendor-verified vsftpd build and verify package che
PASS: CVE knowledge base contains expected information.
SECURITY SUMMARY FOR 10.0.0.30
Total exposed services: 3

Port: 21
Service: ftp
Banner: vsftpd 2.3.4
Potential concerns: FTP may transmit credentials without encryption.
Recommendations: Prefer SFTP or another encrypted transfer protocol.
-----------------------------------

In [38]:

assert "SECURITY SUMMARY" in result
assert "Total exposed services: 3" in result
assert "vsftpd 2.3.4" in result
assert "OpenSSH 7.2p2" in result
assert "Apache Tomcat 9.0.30" in result
assert "LIMITATION" in result

print("PASS: Custom security analysis tool works!")


PASS: Custom security analysis tool works!


In [39]:

import importlib
import project_mcp_server

importlib.reload(project_mcp_server)

service = "Apache Tomcat 9.0.30"

print("BEFORE ADDING KNOWLEDGE\n")

print("Original lookup:")
print(project_mcp_server.lookup_cve(service))

print("\nVersion-aware lookup:")
print(project_mcp_server.lookup_cve_v2(service))


MCP successfully imported!
PASS: Server file has valid Python syntax
OpenSSH 7.2p2 Ubuntu: MATCH
Redis key-value store 4.0.9: MATCH
Linux telnetd: MATCH
Apache httpd 2.2.8: NO MATCH
Knowledge base loaded successfully!
Number of characters: 5435

First 400 characters:

# vsftpd 2.3.4
The vsftpd 2.3.4 FTP server (commonly seen on port 21) shipped with a malicious backdoor introduced into the source distribution. Connecting with a username ending in ":)" opens a command shell on TCP port 6200, granting remote command execution as root. Tracked as CVE-2011-2523. Severity: critical. Remediation: upgrade to a clean, vendor-verified vsftpd build and verify package che
PASS: CVE knowledge base contains expected information.
BEFORE ADDING KNOWLEDGE

Original lookup:
No known CVEs for 'Apache Tomcat 9.0.30' in the knowledge base.

Version-aware lookup:
No known CVEs for 'Apache Tomcat 9.0.30' in the knowledge base.


In [40]:

from pathlib import Path

sources = """# Sources — MCP Cybersecurity Project

## Apache Tomcat 9.0.30

**CVE:** CVE-2020-1938

**Product:** Apache Tomcat 9.0.30

**Vulnerability:** AJP Request Injection and potential Remote
Code Execution (Ghostcat).

### Source 1 — Apache Software Foundation

Apache Tomcat 9 Security Vulnerabilities.

https://tomcat.apache.org/security-9.html

The Apache security advisory documents the affected versions,
AJP-related attack conditions, mitigations, and fixed releases.

### Source 2 — NIST National Vulnerability Database

CVE-2020-1938.

https://nvd.nist.gov/vuln/detail/CVE-2020-1938

The NVD record provides vulnerability information and
affected-version details.

### Source 3 — CVE Program

CVE-2020-1938.

https://www.cve.org/CVERecord?id=CVE-2020-1938

This provides the corresponding official CVE record.

## Project limitations

The services and hosts used in this project are simulated.
The knowledge base is a limited educational dataset,
not a comprehensive vulnerability intelligence system.
"""

Path("/content/SOURCES.md").write_text(
    sources,
    encoding="utf-8"
)

print("PASS: SOURCES.md created successfully")


PASS: SOURCES.md created successfully


In [41]:

import json
from pathlib import Path

custom_expectations = {
    "host": "10.0.0.30",
    "services": [
        {
            "service": "vsftpd 2.3.4",
            "expected": "vsftpd 2.3.4"
        },
        {
            "service": "OpenSSH 7.2p2 Ubuntu",
            "expected": "OpenSSH 7.2p2"
        },
        {
            "service": "Apache Tomcat 9.0.30",
            "expected": "Apache Tomcat 9.0.30"
        }
    ]
}

Path("/content/expected_mine.json").write_text(
    json.dumps(custom_expectations, indent=2),
    encoding="utf-8"
)

print("PASS: Custom ground-truth file created")
print(json.dumps(custom_expectations, indent=2))


PASS: Custom ground-truth file created
{
  "host": "10.0.0.30",
  "services": [
    {
      "service": "vsftpd 2.3.4",
      "expected": "vsftpd 2.3.4"
    },
    {
      "service": "OpenSSH 7.2p2 Ubuntu",
      "expected": "OpenSSH 7.2p2"
    },
    {
      "service": "Apache Tomcat 9.0.30",
      "expected": "Apache Tomcat 9.0.30"
    }
  ]
}


In [42]:

import importlib
import project_mcp_server

importlib.reload(project_mcp_server)

service = "Apache Tomcat 9.0.30"

result = project_mcp_server.lookup_cve_v2(service)

print("AFTER ADDING KNOWLEDGE\n")
print(result)

assert "CVE-2020-1938" in result
assert "Apache Tomcat 9.0.30" in result

print("\nPASS: Tomcat vulnerability successfully retrieved!")


MCP successfully imported!
PASS: Server file has valid Python syntax
OpenSSH 7.2p2 Ubuntu: MATCH
Redis key-value store 4.0.9: MATCH
Linux telnetd: MATCH
Apache httpd 2.2.8: NO MATCH
Knowledge base loaded successfully!
Number of characters: 6334

First 400 characters:

# vsftpd 2.3.4
The vsftpd 2.3.4 FTP server (commonly seen on port 21) shipped with a malicious backdoor introduced into the source distribution. Connecting with a username ending in ":)" opens a command shell on TCP port 6200, granting remote command execution as root. Tracked as CVE-2011-2523. Severity: critical. Remediation: upgrade to a clean, vendor-verified vsftpd build and verify package che
PASS: CVE knowledge base contains expected information.
AFTER ADDING KNOWLEDGE

Apache Tomcat 9.0.30

Apache Tomcat 9.0.30 is within the affected version range for
CVE-2020-1938 (Ghostcat), an AJP request injection vulnerability.

Affected versions: Apache Tomcat 9.0.0.M1 through 9.0.30.

The vulnerability may allow an attacker 

In [43]:

from pathlib import Path

required = [
    "project_mcp_server.py",
    "project_cve_kb.md",
    "expected_cves.json",
    "expected_mine.json",
    "SOURCES.md"
]

for filename in required:
    path = Path("/content") / filename
    print(
        f"{'PASS' if path.exists() else 'FAIL'}: {filename}"
    )

assert all(
    (Path("/content") / name).exists()
    for name in required
)

print("\nAll Phase 2 knowledge-base files are present.")


PASS: project_mcp_server.py
PASS: project_cve_kb.md
PASS: expected_cves.json
PASS: expected_mine.json
PASS: SOURCES.md

All Phase 2 knowledge-base files are present.


In [44]:
GROUND_TRUTH = {s["service"]: s for s in json.load(open(EXPECTED_FILE))["services"]}


def classify(answer, service):
    """GIVEN: sort one tool answer into a bucket, given the ground truth.

    Note the `must_not` check: an answer can contain the right advisory AND an
    advisory for a version the host is not running. That still counts as a wrong
    version -- a report that names a vulnerability which is not there costs an
    analyst just as much time as one that misses a real one.
    """
    truth = GROUND_TRUTH.get(service, {})
    expected = truth.get("expected")
    said_nothing = answer.startswith("No known CVEs")
    if expected is None:
        return "true gap" if said_nothing else "wrong version"
    if said_nothing:
        return "blind spot"
    for forbidden in truth.get("must_not", []):
        if forbidden.lower() in answer.lower():
            return "wrong version"
    return "grounded" if expected.lower() in answer.lower() else "wrong version"


BUCKETS = ["grounded", "blind spot", "wrong version", "true gap"]
summary = {}
for tool in ["lookup_cve", "lookup_cve_v2"]:
    counts = dict.fromkeys(BUCKETS, 0)
    for r in rows:
        r[tool + "_verdict"] = classify(r[tool], r["banner"])
        counts[r[tool + "_verdict"]] += 1
    summary[tool] = counts

print("%-16s %-9s %-11s %-14s %s" % ("tool", "grounded", "blind spot", "wrong version", "true gap"))
for tool, c in summary.items():
    print("%-16s %-9d %-11d %-14d %d"
          % (tool, c["grounded"], c["blind spot"], c["wrong version"], c["true gap"]))

print("\nwhere they disagree:")
print("%-38s %-15s %s" % ("service", "lookup_cve", "lookup_cve_v2"))
for r in rows:
    if r["lookup_cve_verdict"] != r["lookup_cve_v2_verdict"]:
        print("%-38s %-15s %s" % (r["banner"][:36], r["lookup_cve_verdict"], r["lookup_cve_v2_verdict"]))

with open("results.csv", "w", encoding="utf-8") as f:
    f.write("host,port,service,lookup_cve,lookup_cve_v2\n")
    for r in rows:
        f.write("%s,%s,\"%s\",%s,%s\n" % (r["host"], r["port"], r["banner"],
                                          r["lookup_cve_verdict"], r["lookup_cve_v2_verdict"]))
print("\nwrote results.csv")

tool             grounded  blind spot  wrong version  true gap
lookup_cve       6         2           1              2
lookup_cve_v2    9         0           0              2

where they disagree:
service                                lookup_cve      lookup_cve_v2
Linux telnetd                          blind spot      grounded
Apache httpd 2.2.8 (OpenSSL 1.0.1)     wrong version   grounded
Redis key-value store 4.0.9            blind spot      grounded

wrote results.csv


## Part 6 — Close the real gaps  *(edit `project_cve_kb.md`, then re-run)*

Two services come back honest-but-empty: **nginx 1.18.0** and **PostgreSQL 9.6.24**. Better matching
cannot help — the knowledge base has nothing to match. This is the other half of a tool: its data.

**Add two sections to `project_cve_kb.md`**, in the same shape as the ones already there:

```markdown
# nginx 1.18.0
One or two sentences: what the software is, a real CVE or CWE that affects this
version, its severity, the fix, and the port it usually listens on.
```

Then re-run Parts 4 and 5. `true gap` should fall to 0 and `grounded` rise to 11/11.

> **Now be careful, because this lever cuts both ways.** Write a *vague* entry — a bare `# nginx`
> with no version — and it will match every nginx banner ever, including versions that were never
> vulnerable. Try it deliberately: make one entry too generic, re-run, and watch a `wrong version`
> appear. Report what you saw. **A tool is its logic, its data, and its docstring — and all three
> can lie.**

In [45]:
rows = run_async(sweep(HOSTS, ["lookup_cve", "lookup_cve_v2"]))

print("%-16s %-9s %-11s %-14s %s" % ("tool", "grounded", "blind spot", "wrong version", "true gap"))
for tool in ["lookup_cve", "lookup_cve_v2"]:
    counts = dict.fromkeys(BUCKETS, 0)
    for r in rows:
        counts[classify(r[tool], r["banner"])] += 1
    print("%-16s %-9d %-11d %-14d %d"
          % (tool, counts["grounded"], counts["blind spot"],
             counts["wrong version"], counts["true gap"]))

print("\n(If nothing changed, you have not saved project_cve_kb.md yet — the server")
print(" re-reads it on every call, so no restart is needed.)")

tool             grounded  blind spot  wrong version  true gap
lookup_cve       8         2           2              2
lookup_cve_v2    11        0           1              2

(If nothing changed, you have not saved project_cve_kb.md yet — the server
 re-reads it on every call, so no restart is needed.)


## Part 7 — Audit what the server exposes  *(no TODO)*

One last MCP-specific question, straight from the Lecture 3 safety section. An agent will call
anything a server advertises, so **the advertised surface is the attack surface**. Look at what this
server hands to any client that connects.

In [46]:
async def audit():
    async with stdio_client(server, errlog=ERRLOG) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()
            tools = (await session.list_tools()).tools
            resources = (await session.list_resources()).resources
            return tools, resources


tools, resources = run_async(audit())
print("This server advertises %d tools and %d resource(s):\n" % (len(tools), len(resources)))
for t in tools:
    params = list((t.inputSchema or {}).get("properties", {}))
    doc = (t.description or "").strip().splitlines()
    print("  %-16s params=%-22s doc lines=%d" % (t.name, ",".join(params) or "none", len(doc)))
    print("      %s" % (doc[0] if doc else "(NO DESCRIPTION -- an agent is guessing)"))
for r in resources:
    print("  resource %s  (%s)" % (r.uri, r.name))

print("\nFor your write-up, answer these from the output above:")
for q in ["which tools take free-text input from the model, and what would a bad value do?",
          "does any tool touch the network or the filesystem, and how would a client know?",
          "is any description vague enough that an agent could call the wrong tool?",
          "what would you refuse to expose over MCP at all?"]:
    print("  -", q)

This server advertises 5 tools and 1 resource(s):

  port_scan        params=target                 doc lines=4
      Run a (simulated) network port scan of a target host and return open ports
  list_targets     params=none                   doc lines=1
      List the demo hosts this server can scan (simulated lab environment).
  summarize_host_security params=target                 doc lines=16
      Summarize the exposed services and potential security concerns
  lookup_cve       params=service                doc lines=7
      Look up known vulnerabilities for a service or product name (e.g.
  lookup_cve_v2    params=service                doc lines=5
      Look up known vulnerabilities for a service, matching on PRODUCT and
  resource kb://cve  (cve_kb)

For your write-up, answer these from the output above:
  - which tools take free-text input from the model, and what would a bad value do?
  - does any tool touch the network or the filesystem, and how would a client know?
  - is an

## Part 8: your own server, your own tool, your own measurement

Everything above used the provided server and the provided lab, and it was practice. The rest of this
notebook is your own, and no code is provided for it.

First extend the server. In `project_mcp_server.py`:

1. Add one host of your own to `_SIMULATED_HOSTS`, with three realistic service banners, product and
   version in nmap style. Choose them so at least one is covered by your knowledge base and at least
   one is not.
2. Add a tool of your own that an analyst would find useful, for example one that reports the
   severity of a finding, checks a port against a policy list, or summarizes what a host exposes.
   Write its docstring as carefully as its code: the docstring is what a client and a model read.
3. Add the missing knowledge to `project_cve_kb.md` for the services that return nothing, with real
   products and real CVEs, cited in `SOURCES.md`, and record the expected section for each of your
   services in `expected_cves.json`.

Then write the client cells below to:

4. Connect to your server, discover the tools, and show that your new tool is advertised.
5. Sweep your own host with both lookup tools and with your new tool, and sort every answer into the
   four buckets: grounded, blind spot, wrong version, and true gap. Save the table as
   `results_mine.csv`.
6. Make one knowledge base entry deliberately vague by dropping the version, run the sweep again, and
   record what changed.

> When you are finished, run the whole notebook top to bottom and save it with the outputs showing,
> then commit it, the server, the knowledge base, and your results to your private GitHub repository.
> See `MiniProject_MCP.md` section 6.1 for the folder structure.

In [47]:

# PHASE 2: CONNECT TO THE EXTENDED MCP SERVER

async def test_custom_server():
    async with stdio_client(server, errlog=ERRLOG) as (read, write):
        async with ClientSession(read, write) as session:

            await session.initialize()

            # Discover all advertised tools.
            tools_result = await session.list_tools()
            tools = tools_result.tools

            print("DISCOVERED MCP TOOLS")
            print("=" * 55)

            for tool in tools:
                print(f"\nTool: {tool.name}")
                print(f"Description: {tool.description}")

            # Find our custom security analysis tool.
            custom_tool = next(
                (t for t in tools
                 if t.name == "summarize_host_security"),
                None
            )

            assert custom_tool is not None, (
                "Custom security tool was not discovered!"
            )

            print("\nCUSTOM TOOL DISCOVERY: PASS")

            # Call the custom tool through MCP.
            result = await session.call_tool(
                "summarize_host_security",
                {"target": "10.0.0.30"}
            )

            assert not result.isError
            security_report = result.content[0].text

            print("\nCUSTOM SECURITY REPORT")
            print("=" * 55)
            print(security_report)

            # Verify the newly added CVE knowledge.
            cve_result = await session.call_tool(
                "lookup_cve_v2",
                {"service": "Apache Tomcat 9.0.30"}
            )

            assert not cve_result.isError
            cve_text = cve_result.content[0].text

            print("\nTOMCAT VULNERABILITY LOOKUP")
            print("=" * 55)
            print(cve_text)

            return tools, security_report, cve_text


tools, security_report, cve_text = run_async(
    test_custom_server()
)


DISCOVERED MCP TOOLS

Tool: port_scan
Description: Run a (simulated) network port scan of a target host and return open ports
with detected service names and versions, in nmap style.

SIMULATED: returns canned data for known lab hosts; no packets are sent.


Tool: list_targets
Description: List the demo hosts this server can scan (simulated lab environment).

Tool: summarize_host_security
Description: Summarize the exposed services and potential security concerns
for a simulated lab host.

Args:
    target: IP address of a host in the simulated environment.

Returns:
    A readable summary containing the host, number of exposed
    services, open ports, detected software, potential security
    concerns, and recommended defensive actions.

Limitations:
    Uses simulated service banners only. Does not scan a real
    network, validate vulnerabilities, establish exploitability,
    or determine whether a host is secure.
    An unrecognized host is not evidence of zero open ports.


Tool

In [48]:

# FINAL VERIFICATION

tool_names = [tool.name for tool in tools]

expected_tools = [
    "port_scan",
    "list_targets",
    "lookup_cve",
    "lookup_cve_v2",
    "summarize_host_security"
]

for name in expected_tools:
    assert name in tool_names, f"Missing MCP tool: {name}"

assert "Total exposed services: 3" in security_report
assert "vsftpd 2.3.4" in security_report
assert "OpenSSH 7.2p2" in security_report
assert "Apache Tomcat 9.0.30" in security_report
assert "CVE-2020-1938" in cve_text

custom_tool = next(
    tool for tool in tools
    if tool.name == "summarize_host_security"
)

assert custom_tool.description
assert "simulated" in custom_tool.description.lower()

print("PASS: All five MCP tools discovered")
print("PASS: Custom tool description advertised")
print("PASS: Custom security report generated")
print("PASS: All three simulated services identified")
print("PASS: Apache Tomcat CVE retrieved")
print("\nPHASE 2 VERIFICATION SUCCESSFUL!")


PASS: All five MCP tools discovered
PASS: Custom tool description advertised
PASS: Custom security report generated
PASS: All three simulated services identified
PASS: Apache Tomcat CVE retrieved

PHASE 2 VERIFICATION SUCCESSFUL!


### Your measurement

In [49]:

# BASELINE MEASUREMENT

CUSTOM_HOST = "10.0.0.30"

# Use the sweep() function from Part 4.
baseline_rows = run_async(
    sweep(
        [CUSTOM_HOST],
        ["lookup_cve", "lookup_cve_v2"]
    )
)

print("BASELINE AUDIT")
print("=" * 65)

for row in baseline_rows:
    print(f"\nService: {row['banner']}")
    print(f"Port: {row['port']}")

    for tool in ["lookup_cve", "lookup_cve_v2"]:
        answer = row[tool]

        if answer.startswith("No known CVEs"):
            print(f"{tool}: NO ADVISORY")
        else:
            print(f"{tool}: ADVISORY RETURNED")

print("\nTotal services:", len(baseline_rows))


BASELINE AUDIT

Service: vsftpd 2.3.4
Port: 21
lookup_cve: ADVISORY RETURNED
lookup_cve_v2: ADVISORY RETURNED

Service: OpenSSH 7.2p2 Ubuntu
Port: 22
lookup_cve: ADVISORY RETURNED
lookup_cve_v2: ADVISORY RETURNED

Service: Apache Tomcat 9.0.30
Port: 8080
lookup_cve: ADVISORY RETURNED
lookup_cve_v2: ADVISORY RETURNED

Total services: 3


In [50]:

import json
from collections import Counter

# Load original instructor ground truth.
with open(EXPECTED_FILE, encoding="utf-8") as f:
    original_truth = json.load(f)

GROUND_TRUTH = {
    item["service"]: item
    for item in original_truth["services"]
}

# Add expectations for our custom host.
with open("/content/expected_mine.json", encoding="utf-8") as f:
    custom_truth = json.load(f)

for item in custom_truth["services"]:
    GROUND_TRUTH[item["service"]] = item

print("Ground-truth records loaded:", len(GROUND_TRUTH))


Ground-truth records loaded: 12


In [51]:

# Classify the lookup results for our custom host.

BUCKETS = [
    "grounded",
    "blind spot",
    "wrong version",
    "true gap"
]

baseline_summary = {}

for tool in ["lookup_cve", "lookup_cve_v2"]:

    classifications = [
        classify(row[tool], row["banner"])
        for row in baseline_rows
    ]

    counts = Counter(classifications)

    baseline_summary[tool] = counts

    print(f"\n{tool}")
    print("-" * 40)

    for bucket in BUCKETS:
        print(f"{bucket}: {counts[bucket]}")



lookup_cve
----------------------------------------
grounded: 3
blind spot: 0
wrong version: 0
true gap: 0

lookup_cve_v2
----------------------------------------
grounded: 3
blind spot: 0
wrong version: 0
true gap: 0


In [52]:

import pandas as pd

records = []

for row in baseline_rows:
    for tool in ["lookup_cve", "lookup_cve_v2"]:

        records.append({
            "host": row["host"],
            "port": row["port"],
            "service": row["banner"],
            "tool": tool,
            "classification": classify(
                row[tool],
                row["banner"]
            )
        })

baseline_df = pd.DataFrame(records)

display(baseline_df)


,host,port,service,tool,classification
0,10.0.0.30,21,vsftpd 2.3.4,lookup_cve,grounded
1,10.0.0.30,21,vsftpd 2.3.4,lookup_cve_v2,grounded
2,10.0.0.30,22,OpenSSH 7.2p2 Ubuntu,lookup_cve,grounded
3,10.0.0.30,22,OpenSSH 7.2p2 Ubuntu,lookup_cve_v2,grounded
4,10.0.0.30,8080,Apache Tomcat 9.0.30,lookup_cve,grounded
5,10.0.0.30,8080,Apache Tomcat 9.0.30,lookup_cve_v2,grounded


In [53]:

assert len(baseline_rows) == 3
assert len(baseline_df) == 6

for tool in ["lookup_cve", "lookup_cve_v2"]:
    assert sum(baseline_summary[tool].values()) == 3

print("PASS: Three custom services audited")
print("PASS: Both CVE lookup tools evaluated")
print("PASS: All six results classified")


PASS: Three custom services audited
PASS: Both CVE lookup tools evaluated
PASS: All six results classified


In [54]:

# PHASE 3 — FULL MCP SECURITY AUDIT

ALL_HOSTS = [
    "10.0.0.5",
    "10.0.0.8",
    "10.0.0.12",
    "10.0.0.20",
    "10.0.0.30"
]

TOOLS = ["lookup_cve", "lookup_cve_v2"]

# Run the MCP client sweep.
full_rows = run_async(sweep(ALL_HOSTS, TOOLS))

print("FULL MCP SECURITY AUDIT")
print("=" * 50)
print("Hosts audited:", len(ALL_HOSTS))
print("Services discovered:", len(full_rows))
print("Lookup tools evaluated:", len(TOOLS))

assert len(full_rows) == 14
print("\nPASS: Full simulated lab sweep completed")


FULL MCP SECURITY AUDIT
Hosts audited: 5
Services discovered: 14
Lookup tools evaluated: 2

PASS: Full simulated lab sweep completed


In [55]:

import pandas as pd
from collections import Counter

audit_records = []

for row in full_rows:
    for tool in TOOLS:

        verdict = classify(
            row[tool],
            row["banner"]
        )

        audit_records.append({
            "host": row["host"],
            "port": row["port"],
            "service": row["banner"],
            "tool": tool,
            "classification": verdict
        })

audit_df = pd.DataFrame(audit_records)

display(audit_df)


,host,port,service,tool,classification
0,10.0.0.5,21,vsftpd 2.3.4,lookup_cve,grounded
1,10.0.0.5,21,vsftpd 2.3.4,lookup_cve_v2,grounded
2,10.0.0.5,22,OpenSSH 7.2p2 Ubuntu,lookup_cve,grounded
3,10.0.0.5,22,OpenSSH 7.2p2 Ubuntu,lookup_cve_v2,grounded
4,10.0.0.5,80,Apache httpd 2.4.49,lookup_cve,grounded
5,10.0.0.5,80,Apache httpd 2.4.49,lookup_cve_v2,grounded
6,10.0.0.8,139,Samba smbd 3.5.0,lookup_cve,grounded
7,10.0.0.8,139,Samba smbd 3.5.0,lookup_cve_v2,grounded
8,10.0.0.8,3306,MySQL 5.5.40,lookup_cve,grounded
9,10.0.0.8,3306,MySQL 5.5.40,lookup_cve_v2,grounded


In [56]:

comparison = pd.crosstab(
    audit_df["tool"],
    audit_df["classification"]
)

# Include all four categories, even when counts are zero.
comparison = comparison.reindex(
    columns=BUCKETS,
    fill_value=0
)

comparison["total"] = comparison.sum(axis=1)

comparison["grounded_rate"] = (
    comparison["grounded"] /
    comparison["total"] * 100
).round(2)

print("CVE LOOKUP PERFORMANCE COMPARISON")
display(comparison)


CVE LOOKUP PERFORMANCE COMPARISON


classification,grounded,blind spot,wrong version,true gap,total,grounded_rate
tool,,,,,,
lookup_cve,9,2,1,2,14,64.29
lookup_cve_v2,12,0,0,2,14,85.71


In [57]:

difference_records = []

for row in full_rows:
    original = classify(
        row["lookup_cve"],
        row["banner"]
    )

    improved = classify(
        row["lookup_cve_v2"],
        row["banner"]
    )

    if original != improved:
        difference_records.append({
            "host": row["host"],
            "service": row["banner"],
            "original": original,
            "improved": improved
        })

differences_df = pd.DataFrame(
    difference_records,
    columns=["host", "service", "original", "improved"]
)

print("SERVICES WHERE THE TOOLS DISAGREE")
display(differences_df)


SERVICES WHERE THE TOOLS DISAGREE


,host,service,original,improved
0,10.0.0.12,Linux telnetd,blind spot,grounded
1,10.0.0.12,Apache httpd 2.2.8 (OpenSSL 1.0.1),wrong version,grounded
2,10.0.0.12,Redis key-value store 4.0.9,blind spot,grounded


In [58]:

from pathlib import Path

baseline_path = Path("/content/results_baseline.csv")

audit_df.to_csv(baseline_path, index=False)

assert baseline_path.exists()

print("PASS: Baseline results saved")
print("File:", baseline_path.name)
print("Records:", len(audit_df))


PASS: Baseline results saved
File: results_baseline.csv
Records: 28


In [59]:

import pandas as pd
from pathlib import Path

KB_FILE = Path("/content/project_cve_kb.md")
CUSTOM_HOST = "10.0.0.30"
TOOLS = ["lookup_cve", "lookup_cve_v2"]

# Use the original ground truth and custom expectations
# loaded during the previous steps.
def measure(hosts, experiment):
    rows = run_async(sweep(hosts, TOOLS))
    records = []

    for row in rows:
        for tool in TOOLS:
            records.append({
                "experiment": experiment,
                "host": row["host"],
                "port": row["port"],
                "service": row["banner"],
                "tool": tool,
                "classification": classify(
                    row[tool], row["banner"]
                )
            })

    return pd.DataFrame(records)


# 1. Save original knowledge base.
original_kb = KB_FILE.read_text(encoding="utf-8")

assert "# Apache httpd 2.4.49" in original_kb

# 2. Measure before modification.
before = measure(ALL_HOSTS, "before")

try:
    # 3. Make one knowledge-base heading vague.
    vague_kb = original_kb.replace(
        "# Apache httpd 2.4.49",
        "# Apache httpd",
        1
    )
    KB_FILE.write_text(vague_kb, encoding="utf-8")

    # 4. Rerun the MCP audit.
    after = measure(ALL_HOSTS, "after")

finally:
    # 5. Always restore original knowledge base.
    KB_FILE.write_text(original_kb, encoding="utf-8")


# 6. Combine measurements.
results = pd.concat(
    [before, after],
    ignore_index=True
)

# 7. Save the required CSV file.
results.to_csv(
    "/content/results_mine.csv",
    index=False
)

# 8. Display comparison.
summary = pd.crosstab(
    [results["experiment"], results["tool"]],
    results["classification"]
).reindex(
    columns=BUCKETS,
    fill_value=0
)

print("BEFORE VS AFTER VAGUE KNOWLEDGE EXPERIMENT")
display(summary)

print("\nCSV saved: /content/results_mine.csv")
print("Original knowledge base restored:",
      KB_FILE.read_text(encoding="utf-8") == original_kb)


BEFORE VS AFTER VAGUE KNOWLEDGE EXPERIMENT


classification            grounded  blind spot  wrong version  true gap
experiment tool                                                        
after      lookup_cve            9           2              1         2
           lookup_cve_v2        11           0              1         2
before     lookup_cve            9           2              1         2
           lookup_cve_v2        12           0              0         2


CSV saved: /content/results_mine.csv
Original knowledge base restored: True


---

## What to write up

One PDF report and a video of five to ten minutes. See `MiniProject_MCP.md` for the full
requirements. The questions that matter:

- The blind spot is the dangerous bucket. A tool that says no known CVEs about a service the
  knowledge base covers is worse than an error message. Why, and what would a client have to do to
  notice?
- A wrong version answer looks authoritative. Which of the two failures would you rather ship, and
  what does your answer say about how much an agent should trust a tool result?
- Two levers. What did better matching fix, what needed new knowledge, and what broke when you made
  an entry vague on purpose? Give the numbers before and after.
- The docstring is the contract. Which description on your server would you rewrite before letting a
  model use it, and which tool would you refuse to expose at all?
- A tool over MCP compared with a tool inside your agent. In the LangGraph module a tool was a Python
  function you called yourself. Here it is a separate process a client discovers. What did you gain,
  what did it cost, and when would you keep the function?

Submit one PDF report with the link to your project folder and the link to your video on its first
page. The notebook, the server, the knowledge base, and your results live in the repository, which is
private, with your instructor and the TA added as collaborators. The notebook must be committed with
its outputs showing.